# Notebook 11: When To Use SVMs

**Difficulty:** ⭐⭐⭐ | **Estimated time:** 2 hours  
**Theme:** Email / Text Classification

---

## Real-World Analogy First

Every tool has its domain. A scalpel is perfect for surgery but useless for carpentry. A chainsaw handles lumber but has no place in an operating room. Machine learning algorithms are the same — no single model wins every battle.

**SVMs shine in specific scenarios and struggle in others.** Knowing *when* to reach for an SVM — versus logistic regression, random forest, or a neural network — is as important as knowing *how* it works. This notebook builds that judgment.

Think of a spam filter team at an email company. They receive millions of emails per day, but their *labeled training set* is carefully curated: 5,000 expert-reviewed emails. The feature space is huge: TF-IDF over 10,000 words. This is exactly the scenario where SVMs thrive.

---

## Learning Objectives

1. Identify when SVMs outperform alternatives (dataset size, dimensionality, kernel need)
2. Recognize SVM failure modes: scale, imbalance, large n, multi-class
3. Demonstrate the critical importance of feature scaling
4. Compare training time and accuracy across four classifiers
5. Calibrate SVM outputs to probabilities using Platt scaling

## When SVMs Excel

### 1. Small to Medium Datasets (n < 100,000)
SVM training solves a quadratic program (QP). Naive QP is **O(n³)**; sklearn's `libsvm` is roughly **O(n² · p)** depending on kernel. This is:
- Fast and highly effective for **n < 10,000**
- Still competitive up to **n ≈ 100,000**
- Prohibitively slow beyond **n > 1,000,000**

For very large n: prefer SGD classifier, logistic regression, or neural networks.

### 2. High-Dimensional Sparse Features (Text, Genomics)
Linear SVM is excellent for **TF-IDF text classification**. When p >> n (more features than samples), Ridge/Lasso also work, but `LinearSVC` is often faster and equally accurate. The linear kernel only stores support vectors → memory efficient.

### 3. Well-Defined Margin Problems
When class separation is clean, the hard-margin SVM finds a very robust boundary. Before deep learning, **SVM was state-of-the-art for image classification** (with HOG features).

### 4. Kernel Expressivity
When you have domain knowledge about similarity (e.g., **string kernels** for DNA sequences, **graph kernels** for molecules), SVMs accept custom kernels. Any PSD Gram matrix works.

---

## When SVMs Struggle

| Problem | Why SVM Struggles | Better Alternative |
|---|---|---|
| n > 100,000 | QP solver too slow | SGD, Logistic Regression, GBM |
| Multi-class K > 5 | OvO = K(K-1)/2 classifiers | Random Forest, GBM |
| Imbalanced classes | Margin ignores class freq | SVC(class_weight='balanced') |
| Need probabilities | Outputs decision scores, not P | Logistic Regression, calibrated SVM |
| Interpretability | Non-linear kernels are opaque | Decision Trees, Logistic Regression |

---

## Feature Scaling Is Non-Negotiable

SVM is distance-based: it minimizes **||w||²** subject to margin constraints. Features with large magnitudes dominate the distance calculation, forcing the hyperplane toward artificially "important" dimensions.

- Feature A in [0, 1]: contributes ≤ 1 to distance
- Feature B in [0, 1000]: contributes up to 1,000,000 to squared distance

**Always apply `StandardScaler()` before SVM.** Failing to scale causes accuracy collapse.

In [ ]:
# ── Section 0: Imports & Global Settings ──────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.svm import SVC, LinearSVC
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    precision_recall_fscore_support, brier_score_loss
)
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
from sklearn.model_selection import train_test_split

np.random.seed(42)
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 11

print('Libraries loaded successfully.')
print(f'NumPy {np.__version__} | Pandas {pd.__version__}')

## Section 1: Simulated Email Spam Dataset

We simulate a realistic scenario: **5,000 emails**, each represented as a **1,000-dimensional TF-IDF vector** (sparse, high-dimensional). Labels: spam (1) vs ham (0).

This mirrors production email classification: carefully labeled examples, huge feature space, binary output.

In [ ]:
# ── Section 1: Simulated Email Spam Dataset ───────────────────────────────
np.random.seed(42)

N_SAMPLES = 5000
N_FEATURES = 1000   # TF-IDF vocabulary size
SPAM_RATIO = 0.35   # 35% spam — realistic imbalance

n_spam = int(N_SAMPLES * SPAM_RATIO)
n_ham  = N_SAMPLES - n_spam

# Spam emails: slightly elevated TF-IDF scores for 'spam words' (first 150 features)
X_spam = np.random.exponential(scale=0.05, size=(n_spam, N_FEATURES))
X_spam[:, :150] += np.random.exponential(scale=0.15, size=(n_spam, 150))  # spam-word boost

# Ham emails: elevated scores for 'normal words' (features 300-600)
X_ham = np.random.exponential(scale=0.03, size=(n_ham, N_FEATURES))
X_ham[:, 300:600] += np.random.exponential(scale=0.12, size=(n_ham, 300))

X = np.vstack([X_spam, X_ham])
y = np.array([1]*n_spam + [0]*n_ham)  # 1=spam, 0=ham

# Shuffle
idx = np.random.permutation(N_SAMPLES)
X, y = X[idx], y[idx]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Dataset: {N_SAMPLES} emails, {N_FEATURES} TF-IDF features')
print(f'Train: {X_train.shape[0]} samples | Test: {X_test.shape[0]} samples')
print(f'Class distribution — spam: {y_train.mean():.1%} | ham: {1-y_train.mean():.1%}')

# Scale features (mandatory for SVM)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

## Section 2: Training Time Benchmark — SVM vs. Alternatives

We compare four classifiers on the same dataset: **SVC (RBF)**, **Logistic Regression**, **Random Forest**, and **Gradient Boosting**. We measure training time and test accuracy to illustrate SVM's speed/accuracy trade-off on a medium-sized dataset.

In [ ]:
# ── Section 2: Training Time Benchmark ───────────────────────────────────
models = {
    'SVC (RBF)':          SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42),
    'LinearSVC':          LinearSVC(C=1.0, max_iter=2000, random_state=42),
    'Logistic Regression':LogisticRegression(C=1.0, max_iter=500, random_state=42),
    'Random Forest':      RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting':  GradientBoostingClassifier(n_estimators=100, random_state=42),
}

results = []
for name, model in models.items():
    t0 = time.perf_counter()
    model.fit(X_train_sc, y_train)
    train_time = time.perf_counter() - t0

    y_pred = model.predict(X_test_sc)
    acc    = accuracy_score(y_test, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='binary')

    results.append({
        'Model': name,
        'Train Time (s)': round(train_time, 3),
        'Accuracy': round(acc, 4),
        'Precision': round(prec, 4),
        'Recall': round(rec, 4),
        'F1': round(f1, 4),
    })
    print(f'{name:25s} | time={train_time:.3f}s | acc={acc:.4f} | F1={f1:.4f}')

df_results = pd.DataFrame(results).set_index('Model')
print('\n', df_results.to_string())

In [ ]:
# ── Visualize benchmark results ───────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

model_names = df_results.index.tolist()
colors = sns.color_palette('husl', len(model_names))

# Plot 1: Training time
axes[0].barh(model_names, df_results['Train Time (s)'], color=colors)
axes[0].set_xlabel('Training Time (seconds)', fontsize=12)
axes[0].set_title('Training Time Comparison\n(n=4000 emails, p=1000 features)', fontsize=13)
for i, v in enumerate(df_results['Train Time (s)']):
    axes[0].text(v + 0.01, i, f'{v:.3f}s', va='center', fontsize=10)

# Plot 2: F1 Score
axes[1].barh(model_names, df_results['F1'], color=colors)
axes[1].set_xlabel('F1 Score (spam class)', fontsize=12)
axes[1].set_title('Classification Performance\n(F1 Score, binary spam detection)', fontsize=13)
axes[1].set_xlim(0.5, 1.05)
for i, v in enumerate(df_results['F1']):
    axes[1].text(v + 0.002, i, f'{v:.4f}', va='center', fontsize=10)

plt.suptitle('SVM vs. Alternatives: Speed/Accuracy Trade-off\nEmail Spam Dataset (5,000 samples, 1,000 TF-IDF features)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('\nKey insight: LinearSVC is often the sweet spot for text — nearly as fast as')
print('Logistic Regression but with strong margin-based generalization.')

## Section 3: Feature Scaling Demo — The Danger of Unscaled Features

SVMs compute distances in feature space. If one feature has values in [0, 1] and another in [0, 1000], the distance metric is completely dominated by the large-magnitude feature. The margin boundary ends up perpendicular to a single axis — ignoring all other information.

We demonstrate this with a 2-feature version of our spam data where:
- Feature A (word frequency): values in [0, 1]
- Feature B (email length in characters): values in [0, 5000]

In [ ]:
# ── Section 3: Feature Scaling Demo ──────────────────────────────────────
np.random.seed(42)

# 2D toy: Feature A (word freq [0,1]), Feature B (email length [0,5000])
n_demo = 300
# Spam: high word freq AND long emails
fa_spam = np.random.uniform(0.6, 1.0, n_demo//2)
fb_spam = np.random.uniform(2500, 5000, n_demo//2)
# Ham: low word freq AND shorter emails
fa_ham  = np.random.uniform(0.0, 0.5, n_demo//2)
fb_ham  = np.random.uniform(0, 2200, n_demo//2)

X2 = np.column_stack([
    np.concatenate([fa_spam, fa_ham]),
    np.concatenate([fb_spam, fb_ham]),
])
y2 = np.array([1]*(n_demo//2) + [0]*(n_demo//2))

X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y2, test_size=0.3, random_state=42)

# Unscaled SVM
svm_unscaled = SVC(kernel='rbf', C=1.0, gamma='scale')
svm_unscaled.fit(X2_train, y2_train)
acc_unscaled = accuracy_score(y2_test, svm_unscaled.predict(X2_test))

# Scaled SVM
scaler2 = StandardScaler()
X2_train_sc = scaler2.fit_transform(X2_train)
X2_test_sc  = scaler2.transform(X2_test)

svm_scaled = SVC(kernel='rbf', C=1.0, gamma='scale')
svm_scaled.fit(X2_train_sc, y2_train)
acc_scaled = accuracy_score(y2_test, svm_scaled.predict(X2_test_sc))

print(f'SVM WITHOUT scaling: accuracy = {acc_unscaled:.4f}')
print(f'SVM WITH scaling:    accuracy = {acc_scaled:.4f}')
print(f'\nAccuracy drop from not scaling: {acc_scaled - acc_unscaled:.4f} ({(acc_scaled-acc_unscaled)/acc_scaled:.1%} relative)')

## Section 4: Scaling Effect Visualization — How Scale Deforms the Margin

In [ ]:
# ── Section 4: Scaling Effect Visualization ───────────────────────────────
def plot_decision_boundary_2d(model, X, y, ax, title, scaler=None,
                               feat_names=('Feature A', 'Feature B')):
    """Plot decision boundary for a 2D SVM. Handles scaled models."""
    x_min, x_max = X[:, 0].min() - 0.05*X[:, 0].ptp(), X[:, 0].max() + 0.05*X[:, 0].ptp()
    y_min, y_max = X[:, 1].min() - 0.05*X[:, 1].ptp(), X[:, 1].max() + 0.05*X[:, 1].ptp()
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                         np.linspace(y_min, y_max, 300))
    grid = np.c_[xx.ravel(), yy.ravel()]
    if scaler is not None:
        grid = scaler.transform(grid)
    Z = model.predict(grid).reshape(xx.shape)

    ax.contourf(xx, yy, Z, alpha=0.25, cmap='RdYlGn')
    ax.contour(xx, yy, Z, colors='k', linewidths=1.5)
    colors = ['#e74c3c', '#2ecc71']
    for cls, c in zip([1, 0], colors):
        mask = y == cls
        ax.scatter(X[mask, 0], X[mask, 1], c=c,
                   label='Spam' if cls==1 else 'Ham',
                   edgecolors='k', linewidths=0.5, s=40, alpha=0.8)
    ax.set_xlabel(feat_names[0])
    ax.set_ylabel(feat_names[1])
    ax.set_title(title, fontweight='bold')
    ax.legend(loc='upper left', fontsize=9)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

plot_decision_boundary_2d(
    svm_unscaled, X2_train, y2_train, axes[0],
    f'UNSCALED Features\nAcc={acc_unscaled:.3f}\nFeature B [0-5000] dominates!',
    scaler=None,
    feat_names=('Word Freq [0–1]', 'Email Length [0–5000]')
)

plot_decision_boundary_2d(
    svm_scaled, X2_train, y2_train, axes[1],
    f'SCALED Features (StandardScaler)\nAcc={acc_scaled:.3f}\nBoth features contribute equally',
    scaler=scaler2,
    feat_names=('Word Freq [0–1]', 'Email Length [0–5000]')
)

plt.suptitle('Feature Scaling Is Critical for SVMs\n'
             'Unscaled: boundary ignores small-magnitude feature | '
             'Scaled: balanced margin uses both features',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nObservation: Without scaling, the decision boundary is nearly horizontal —')
print('it relies only on email length. Word frequency is completely ignored!')

## Section 5: High-Dimensional Text — LinearSVC vs. RBF SVM

Text data is the native habitat of linear SVMs:
- **p >> n**: 1,000 features, 4,000 samples → linear kernel avoids overfitting
- **Sparsity**: TF-IDF vectors are sparse; the linear kernel doesn't need non-linear interactions
- **Speed**: `LinearSVC` is dramatically faster than `SVC(kernel='rbf')` for large p

We compare `LinearSVC` vs `SVC(kernel='rbf')` on our 1,000-feature simulated email dataset.

In [ ]:
# ── Section 5: High-Dimensional Text Classification ───────────────────────
np.random.seed(42)

experiments = [
    ('LinearSVC',      LinearSVC(C=1.0, max_iter=3000)),
    ('SVC (RBF)',      SVC(kernel='rbf', C=1.0, gamma='scale')),
    ('SVC (linear)',   SVC(kernel='linear', C=1.0)),
]

text_results = []
for name, clf in experiments:
    t0 = time.perf_counter()
    clf.fit(X_train_sc, y_train)
    elapsed = time.perf_counter() - t0

    y_pred = clf.predict(X_test_sc)
    acc = accuracy_score(y_test, y_pred)
    _, _, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='binary')

    text_results.append({'Classifier': name, 'Train Time (s)': elapsed,
                          'Accuracy': acc, 'F1': f1})
    print(f'{name:18s} | time={elapsed:.3f}s | acc={acc:.4f} | F1={f1:.4f}')

df_text = pd.DataFrame(text_results).set_index('Classifier')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
clrs = sns.color_palette('Set2', 3)

axes[0].bar(df_text.index, df_text['Train Time (s)'], color=clrs)
axes[0].set_ylabel('Training Time (s)')
axes[0].set_title('Training Time\n(1000 TF-IDF features, n=4000)', fontweight='bold')
axes[0].tick_params(axis='x', rotation=15)
for i, v in enumerate(df_text['Train Time (s)']):
    axes[0].text(i, v + 0.01, f'{v:.3f}s', ha='center', fontsize=10)

axes[1].bar(df_text.index, df_text['F1'], color=clrs)
axes[1].set_ylabel('F1 Score')
axes[1].set_title('F1 Score on Test Set\n(spam detection)', fontweight='bold')
axes[1].set_ylim(0.5, 1.05)
axes[1].tick_params(axis='x', rotation=15)
for i, v in enumerate(df_text['F1']):
    axes[1].text(i, v + 0.005, f'{v:.4f}', ha='center', fontsize=10)

plt.suptitle('LinearSVC vs RBF SVM on High-Dimensional Text\n'
             'Linear kernel is faster AND better for sparse TF-IDF features',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nKey finding: LinearSVC is much faster than SVC(rbf) on text.')
print('For TF-IDF features: always try LinearSVC first.')

## Section 6: SVM Probability Calibration with Platt Scaling

**Problem:** `SVC.predict()` returns class labels. `SVC.decision_function()` returns real-valued scores — not probabilities. A score of +2.5 does NOT mean 97.5% confidence.

**Platt scaling** fits a sigmoid function on top of SVM scores using cross-validation. `SVC(probability=True)` does this automatically, but:
1. It adds 5-fold CV internally → **5× slower training**
2. Platt scaling can still be poorly calibrated if the score distribution isn't sigmoidal

We compare calibration using a **reliability diagram** (calibration curve).

In [ ]:
# ── Section 6: SVM Probability Calibration ────────────────────────────────
np.random.seed(42)

# We use a smaller subset for speed (Platt scaling is slow)
N_CAL = 1000
X_cal_train = X_train_sc[:N_CAL]
y_cal_train = y_train[:N_CAL]
X_cal_test  = X_test_sc[:300]
y_cal_test  = y_test[:300]

# 1. Raw SVC (no probabilities)
svc_raw = SVC(kernel='rbf', C=1.0, gamma='scale')
svc_raw.fit(X_cal_train, y_cal_train)

# 2. SVC with Platt scaling
t0 = time.perf_counter()
svc_platt = SVC(kernel='rbf', C=1.0, gamma='scale', probability=True)
svc_platt.fit(X_cal_train, y_cal_train)
platt_time = time.perf_counter() - t0
print(f'SVC(probability=True) training time: {platt_time:.2f}s  [5-fold CV overhead]')

# 3. Isotonic calibration via CalibratedClassifierCV
svc_base = SVC(kernel='rbf', C=1.0, gamma='scale')
svc_isotonic = CalibratedClassifierCV(svc_base, cv=5, method='isotonic')
svc_isotonic.fit(X_cal_train, y_cal_train)

# 4. Logistic Regression baseline
lr = LogisticRegression(C=1.0, max_iter=500)
lr.fit(X_cal_train, y_cal_train)

# Calibration curves
prob_platt    = svc_platt.predict_proba(X_cal_test)[:, 1]
prob_isotonic = svc_isotonic.predict_proba(X_cal_test)[:, 1]
prob_lr       = lr.predict_proba(X_cal_test)[:, 1]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Reliability diagrams
ax = axes[0]
for probs, label, color in [
    (prob_platt,    'SVC + Platt scaling',   '#e74c3c'),
    (prob_isotonic, 'SVC + Isotonic',         '#3498db'),
    (prob_lr,       'Logistic Regression',    '#2ecc71'),
]:
    frac_pos, mean_pred = calibration_curve(y_cal_test, probs, n_bins=10)
    brier = brier_score_loss(y_cal_test, probs)
    ax.plot(mean_pred, frac_pos, 'o-', label=f'{label} (Brier={brier:.4f})', color=color)

ax.plot([0, 1], [0, 1], 'k--', label='Perfect calibration')
ax.set_xlabel('Mean Predicted Probability')
ax.set_ylabel('Fraction of Positives')
ax.set_title('Reliability Diagram\n(Calibration Curves)', fontweight='bold')
ax.legend(fontsize=9)

# Probability distributions
ax = axes[1]
ax.hist(prob_platt,    bins=30, alpha=0.5, label='SVC + Platt', color='#e74c3c', density=True)
ax.hist(prob_isotonic, bins=30, alpha=0.5, label='SVC + Isotonic', color='#3498db', density=True)
ax.hist(prob_lr,       bins=30, alpha=0.5, label='Logistic Regression', color='#2ecc71', density=True)
ax.set_xlabel('Predicted Probability of Spam')
ax.set_ylabel('Density')
ax.set_title('Predicted Probability Distributions', fontweight='bold')
ax.legend(fontsize=9)

plt.suptitle('SVM Probability Calibration vs. Logistic Regression\n'
             'Lower Brier score = better-calibrated probabilities',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nKey insight: If you need well-calibrated probabilities, Logistic Regression')
print('is often preferable to SVC(probability=True) — it is faster AND better calibrated.')

## Section 7: Class Imbalance — SVC With vs. Without class_weight='balanced'

SVMs optimize the **geometric margin** — they are agnostic to class frequencies. In an imbalanced dataset (e.g., 90% ham, 10% spam), the SVM naturally biases toward the majority class because more ham samples are near the margin.

`class_weight='balanced'` multiplies each class's C by `n_total / (n_classes * n_class_i)`, giving minority class samples more weight in the QP. This forces the margin to respect class rarity.

In [ ]:
# ── Section 7: Class Imbalance Demo ──────────────────────────────────────
np.random.seed(42)

# Create severely imbalanced dataset: 90% ham, 10% spam
N_IMB = 2000
n_sp_imb = int(N_IMB * 0.10)
n_hm_imb = N_IMB - n_sp_imb

X_sp = np.random.exponential(0.05, (n_sp_imb, 50))
X_sp[:, :15] += np.random.exponential(0.18, (n_sp_imb, 15))
X_hm = np.random.exponential(0.03, (n_hm_imb, 50))
X_hm[:, 20:40] += np.random.exponential(0.10, (n_hm_imb, 20))

X_imb = np.vstack([X_sp, X_hm])
y_imb = np.array([1]*n_sp_imb + [0]*n_hm_imb)

idxs = np.random.permutation(N_IMB)
X_imb, y_imb = X_imb[idxs], y_imb[idxs]

Xi_tr, Xi_te, yi_tr, yi_te = train_test_split(X_imb, y_imb, test_size=0.25,
                                                random_state=42, stratify=y_imb)
sc_imb = StandardScaler()
Xi_tr_sc = sc_imb.fit_transform(Xi_tr)
Xi_te_sc = sc_imb.transform(Xi_te)

print(f'Imbalanced dataset: spam={y_imb.mean():.1%} | ham={1-y_imb.mean():.1%}')

svc_no_weight = SVC(kernel='rbf', C=1.0, gamma='scale')
svc_balanced  = SVC(kernel='rbf', C=1.0, gamma='scale', class_weight='balanced')

for name, clf in [('SVC (no weight)', svc_no_weight), ('SVC (balanced)', svc_balanced)]:
    clf.fit(Xi_tr_sc, yi_tr)
    y_pred_i = clf.predict(Xi_te_sc)
    print(f'\n{name}:')
    print(classification_report(yi_te, y_pred_i, target_names=['Ham', 'Spam']))

In [ ]:
# ── Visualize precision/recall for both models ────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (name, clf) in zip(axes, [
    ('SVC (no weight)',   svc_no_weight),
    ('SVC (balanced)',    svc_balanced),
]):
    y_pred_i = clf.predict(Xi_te_sc)
    cm = confusion_matrix(yi_te, y_pred_i)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Pred Ham', 'Pred Spam'],
                yticklabels=['True Ham', 'True Spam'])
    acc_i = accuracy_score(yi_te, y_pred_i)
    _, _, f1_i, _ = precision_recall_fscore_support(yi_te, y_pred_i, average='binary')
    ax.set_title(f'{name}\nAcc={acc_i:.3f} | Spam F1={f1_i:.3f}', fontweight='bold')

plt.suptitle('Class Imbalance: SVC Without vs. With class_weight="balanced"\n'
             '90% Ham / 10% Spam dataset — look at spam recall!',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nWithout class_weight: SVM ignores spam (low recall).')
print('With class_weight="balanced": SVM learns to detect the rare spam class.')

## Section 8: Decision Flowchart — When to Use Which Model

Use this decision guide when choosing a classifier for a new problem.

In [ ]:
# ── Section 8: Decision Flowchart (Matplotlib diagram) ────────────────────
fig, ax = plt.subplots(figsize=(14, 10))
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis('off')

def box(ax, x, y, w, h, text, color='#3498db', fontsize=9, text_color='white'):
    rect = mpatches.FancyBboxPatch((x - w/2, y - h/2), w, h,
                                    boxstyle='round,pad=0.1',
                                    facecolor=color, edgecolor='#2c3e50', linewidth=1.5)
    ax.add_patch(rect)
    ax.text(x, y, text, ha='center', va='center', fontsize=fontsize,
            color=text_color, fontweight='bold', wrap=True,
            multialignment='center')

def arrow(ax, x1, y1, x2, y2, label='', color='#2c3e50'):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color=color, lw=1.8))
    if label:
        mx, my = (x1+x2)/2, (y1+y2)/2
        ax.text(mx+0.08, my, label, fontsize=8, color='#7f8c8d', style='italic')

# Start
box(ax, 5, 9.3, 3.5, 0.7, 'New Classification Problem', color='#2c3e50', fontsize=10)

# Q1: n > 100K?
box(ax, 5, 8.0, 3.5, 0.7, 'n > 100,000 samples?', color='#8e44ad', fontsize=9)
arrow(ax, 5, 8.95, 5, 8.35)

# Yes → SGD/LR
box(ax, 8.5, 8.0, 2.2, 0.65, 'SGD / Logistic Reg\n/ Neural Network', color='#e74c3c', fontsize=8)
arrow(ax, 6.75, 8.0, 7.4, 8.0, 'Yes')

# Q2: High-dim sparse?
box(ax, 5, 6.7, 3.5, 0.7, 'High-dim sparse text?\n(TF-IDF, p >> n)', color='#8e44ad', fontsize=9)
arrow(ax, 5, 7.65, 5, 7.05, 'No')

# Yes → LinearSVC
box(ax, 8.5, 6.7, 2.2, 0.65, 'LinearSVC\nor Logistic Reg', color='#27ae60', fontsize=8)
arrow(ax, 6.75, 6.7, 7.4, 6.7, 'Yes')

# Q3: Need probabilities?
box(ax, 5, 5.4, 3.5, 0.7, 'Need calibrated\nprobabilities?', color='#8e44ad', fontsize=9)
arrow(ax, 5, 6.35, 5, 5.75, 'No')

# Yes → LogReg
box(ax, 8.5, 5.4, 2.2, 0.65, 'Logistic Regression\n(or calibrated SVM)', color='#27ae60', fontsize=8)
arrow(ax, 6.75, 5.4, 7.4, 5.4, 'Yes')

# Q4: Multi-class K>5?
box(ax, 5, 4.1, 3.5, 0.7, 'Multi-class K > 5?', color='#8e44ad', fontsize=9)
arrow(ax, 5, 5.05, 5, 4.45, 'No')

# Yes → RF/GBM
box(ax, 8.5, 4.1, 2.2, 0.65, 'Random Forest\nor GBM', color='#e67e22', fontsize=8)
arrow(ax, 6.75, 4.1, 7.4, 4.1, 'Yes')

# Q5: Domain kernel?
box(ax, 5, 2.8, 3.5, 0.7, 'Domain similarity\nknowledge (custom kernel)?', color='#8e44ad', fontsize=9)
arrow(ax, 5, 3.75, 5, 3.15, 'No')

# Yes → SVM custom kernel
box(ax, 8.5, 2.8, 2.2, 0.65, 'SVM with\nCustom Kernel', color='#27ae60', fontsize=8)
arrow(ax, 6.75, 2.8, 7.4, 2.8, 'Yes')

# Default: SVM (RBF)
box(ax, 5, 1.5, 3.5, 0.9, 'SVM (RBF or Linear)\nn < 100K, binary/few classes\nclean margin', color='#27ae60', fontsize=9)
arrow(ax, 5, 2.45, 5, 1.95, 'No')

ax.set_title('When to Use Which Classifier: Decision Flowchart',
             fontsize=14, fontweight='bold', pad=10)
plt.tight_layout()
plt.show()

## Section 9: Summary Table

| Scenario | Best Choice | Why |
|---|---|---|
| n < 5,000, binary, clean boundaries | **SVM (RBF or linear)** | Excellent generalization, robust margin |
| Text / high-dimensional sparse | **LinearSVC or Logistic Reg** | Fast, p >> n, linear is sufficient |
| n > 100,000 | **SGD, Logistic Reg, GBM** | SVM QP solver too slow |
| Multi-class K > 5 | **Random Forest, GBM** | OvO = K(K-1)/2 SVMs gets unwieldy |
| Need calibrated probabilities | **Logistic Regression** | Native probability output |
| Domain kernel knowledge | **SVM with custom kernel** | Custom PSD Gram matrix |
| Imbalanced classes | **SVC(class_weight='balanced')** | Reweights minority class in QP |

### Golden Rules
1. **Always StandardScaler before SVM** — failure to scale kills performance
2. **LinearSVC >> SVC(kernel='rbf')** for text/TF-IDF
3. **SVM is not for n > 100K** — use SGD or neural networks
4. **For probabilities**: prefer Logistic Regression; if you must use SVM, use `CalibratedClassifierCV`
5. **Imbalanced data**: always try `class_weight='balanced'`

## Self-Check Questions

Answer these questions to test your understanding. Reveal the answer by clicking the triangle.

---

**Q1.** You have 500,000 emails and want to train an SVM. What is the biggest practical obstacle? What alternative would you recommend?

<details>
<summary>Answer</summary>

**Obstacle:** Training time. sklearn's `libsvm` scales approximately O(n²·p) in the worst case. For n=500,000, this involves computing 250 billion distance evaluations — even a single epoch would take hours. The QP solver must also store the kernel matrix (n×n), requiring ~2TB of RAM for 500K samples.

**Recommendations:**
- **SGDClassifier(loss='hinge')**: implements a stochastic online SVM, scales O(n) — trains in seconds
- **Logistic Regression** with L2 regularization: similar accuracy, probabilistic output, scales O(n)
- **Gradient Boosting** (LightGBM/XGBoost): handles large n extremely well
- **LinearSVC** with `dual=False`: uses primal formulation, scales O(n·p) — feasible for large n

For text data specifically: `SGDClassifier(loss='hinge')` is the go-to SVM-like model at large scale.
</details>

---

**Q2.** You forgot to normalize features before SVM. Feature A has values in [0, 1] and Feature B has values in [0, 1000]. How does this affect the margin?

<details>
<summary>Answer</summary>

SVM's margin width is **2 / ||w||** and the constraint is y_i(w·x_i + b) ≥ 1. The weight vector w minimizes ||w||², so in the unscaled case:

- Feature B contributes up to 1000× more to w·x than Feature A
- To satisfy the constraint y_i(w·x_i + b) ≥ 1, the SVM can easily satisfy it using Feature B alone
- Result: w_A ≈ 0 (Feature A is ignored), w_B drives the boundary

The hyperplane becomes nearly perpendicular to Feature B's axis. Feature A — which may contain critical discriminative information — is completely discarded. The decision boundary effectively becomes: `if email_length > threshold: spam`.

After `StandardScaler`: both features have mean=0, std=1. The SVM can now use both equally. The margin becomes geometrically meaningful and uses all available information.
</details>

---

**Q3.** `SVC.predict()` gives class labels. `SVC(probability=True)` with Platt scaling gives probabilities. Why might the Platt probabilities be unreliable, and what's a better alternative?

<details>
<summary>Answer</summary>

**Why Platt scaling can be unreliable:**

1. **Model mismatch**: Platt scaling fits a sigmoid A·f(x) + B to the SVM's decision scores. This assumes decision scores follow a logistic distribution — but SVM scores often don't. The sigmoid is a poor fit for multimodal or skewed score distributions.

2. **Cross-validation artifacts**: `SVC(probability=True)` internally runs 5-fold CV on training data to get out-of-sample scores before fitting the sigmoid. This means only 4/5 of training data was used for each fold's SVM — the final model trained on all data may produce different score distributions.

3. **Overconfidence**: Platt probabilities often cluster near 0 and 1 (see the bimodal histogram), not reflecting true uncertainty.

**Better alternatives:**
- **Isotonic regression calibration**: `CalibratedClassifierCV(svc, method='isotonic')` — non-parametric, doesn't assume sigmoid shape
- **Logistic Regression**: natively outputs well-calibrated probabilities; no extra step needed
- **Temperature scaling**: post-hoc calibration technique

If you need probabilities, Logistic Regression is usually preferable to SVM + Platt scaling.
</details>